# Savor - Inference: per-student recommendations

End-to-end: take **one** before-meal plate photo + a face photo, and produce a
personalised, **unit-counted** ("2 katori", "1 ladle", ...) dinner recommendation
tailored to that student's recent intake.

Uses only the existing held-out test split for inference examples. Grounding is
closed-menu, SAM masks are clipped to their native-image boxes, and the cached
validated LLM response is used for recommendation. Missing current-version test
groundings are requested once with the configured rate-limit buffer.


In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / "src" / "granica").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from granica.config import ROOT, RUNTIME
sys.path.insert(0, str(ROOT / "scripts"))
from granica.io_utils import load_json
from granica.menu import load_menu, menu_for, menu_items
from PIL import Image
from IPython.display import display
import matplotlib.pyplot as plt
menu = load_menu()
print("menu loaded:", RUNTIME.day_of_week, RUNTIME.meal, menu_items(menu, RUNTIME.day_of_week, RUNTIME.meal))


## Step 1 - Identity retrieval

Load the face gallery (built in notebook 1). Run it on a face photo (any
enrollment/ID photo). Plate photos have no face, so a plate is attributed via
its filename stem; for the live demo below we pass the face photo of the student.
Accuracy recap is printed from `artifacts/faces/face_metrics.json`.

In [ ]:
from granica.faceid import FaceIdentifier, parse_plate_filename
from granica.io_utils import load_json

ident = FaceIdentifier(); ident.load_gallery()
print("gallery:", len(ident.names), "persons")

# accuracy recap from the training notebook
metrics = load_json(ROOT / "artifacts/faces/face_metrics.json")
print(f"rank-1 = {metrics['rank1_accuracy_mean']:.3f} | "
      f"same={metrics['verification_same_mean']:.3f} diff={metrics['verification_diff_mean']:.3f} "
      f"| EER={metrics['eer']:.3f}")

# identify a face photo
face_img = sorted(RUNTIME.db_dir.glob("*.jpeg"))[0]
res = ident.identify(face_img)
person = res["best"]
print("face photo ->", person, "score", round(res["score"], 3))

# identify a plate photo via filename convention
plate_img = RUNTIME.plate_images("test", "pre")[0]
person = parse_plate_filename(plate_img.stem)[0]   # name from filename
print("plate", plate_img.name, "-> student:", person)


### Test split - before/after photos

These are every actual paired photo from the configured **test split**. The
inference path uses only these held-out images; no external notebook images are added.

In [ ]:
test_pre = {p.stem: p for p in RUNTIME.plate_images("test", "pre")}
test_post = {p.stem: p for p in RUNTIME.plate_images("test", "post")}
test_ids = sorted(set(test_pre) | set(test_post))
fig, axes = plt.subplots(max(1, len(test_ids)), 2, figsize=(14, 4 * max(1, len(test_ids))), squeeze=False)
for row, person_id in enumerate(test_ids):
    for col, stage in enumerate(("pre", "post")):
        path = (test_pre if stage == "pre" else test_post).get(person_id)
        ax = axes[row][col]
        if path:
            ax.imshow(Image.open(path)); ax.set_title(f"TEST / {stage.upper()} / {person_id}")
        else:
            ax.text(0.5, 0.5, "no paired photo", ha="center", va="center")
        ax.axis("off")
plt.tight_layout(); display(fig); plt.close(fig)


## Step 2 - Ground the food on the plate

OpenAI vision labels visible foods and returns normalized full-image boxes,
restricted to the closed dinner whitelist. The image set is the held-out test split.

In [ ]:
from granica.grounding import OpenAIGrounding
from granica.io_utils import load_json
gmodel = OpenAIGrounding()
test_images = [p for stage in ("pre", "post") for p in RUNTIME.plate_images("test", stage)]
whitelist = menu_items(menu, RUNTIME.day_of_week, RUNTIME.meal)
grounds = {}
for image_path in test_images:
    grounds[str(image_path)] = gmodel.ground(image_path, whitelist)
    print(image_path.name, [x["item"] for x in grounds[str(image_path)]["items"]])
g = grounds[str(plate_img)]
print("items on plate:", [b["item"] for b in g["items"]])


## Step 3 - Segment & predict weight with the fitted model

MobileSAM counts pixels inside each box; the per-category `grams = a*pixels + b`
models from notebook 1 turn those counts into grams.


In [ ]:
from granica.segmentation import SAMSegmenter
from granica.weights import predict_plate_items
models = load_json(ROOT / "artifacts/weights/weight_models.json")

seg = SAMSegmenter()
test_predictions = {}
test_segmentation = {}
for image_path in test_images:
    image_grounding = grounds[str(image_path)]
    image_segmentation = seg.segment(
        image_path, [b["bbox"] for b in image_grounding["items"]],
        labels=[b["item"] for b in image_grounding["items"]])
    test_segmentation[str(image_path)] = image_segmentation
    image_items = [{"item": b["item"], "pixels": sb.get("pixel_count", 0)}
                   for b, sb in zip(image_grounding["items"], image_segmentation["boxes"])]
    test_predictions[f"test/{image_path.parent.name}/{image_path.stem}"] = predict_plate_items(image_items, menu, models)
from granica.io_utils import dump_json
dump_json(test_predictions, ROOT / "artifacts/data/test_predictions.json")
seg_rec = test_segmentation[str(plate_img)]
items = [{"item": b["item"], "pixels": sb.get("pixel_count", 0)}
         for b, sb in zip(g["items"], seg_rec["boxes"])]

pred = predict_plate_items(items, menu, models)
print("predicted plate weight:", pred["total_grams"], "g")
for it in pred["predicted_items"]:
    print(f"  {it['item']:26s} {it['grams']}g")
viz_path = ROOT / "artifacts" / "visualizations" / "segmentation" / plate_img.parent.parent.name / plate_img.parent.name / f"{plate_img.stem}_segmentation.png"
display(Image.open(viz_path))


## Step 4 - Student history + recommendation examples

The selected student's prompt is shown for audit. Validated recommendations for
several students are loaded from the cache so notebook reruns avoid extra
recommendation API calls. Each recommendation is constrained to the menu and
serving-unit table.


In [ ]:
from granica.recommend import recommend, build_recommendation_prompt, validate_and_repair
from granica.io_utils import load_json

people = load_json(ROOT / "dataset/student_history.json")
history = people["people"].get(person, {})

prompt = build_recommendation_prompt(menu, RUNTIME.day_of_week, RUNTIME.meal, history)
print("=== system + user prompt sent to OpenAI ===")
print(prompt[0]["content"][:400], "...\n")
print(prompt[1]["content"][:400], "...")

all_recs = load_json(ROOT / "artifacts/recommendations/recommendations.json")["recommendations"]
rec = all_recs.get(person)
if rec is None:
    raise KeyError(f"No cached LLM recommendation for {person}; run scripts/run_recommendations.py")
print("\n=== recommendation (validated) ===")
print("total:", rec["total_grams"], "g  within band:", rec["within_band"])
for r in rec["recommendations"]:
    print(f"  {r['item']:26s} {r['count']} {r['unit']} = {r['grams']}g")
example_people = [name for name, result in sorted(all_recs.items())
                  if isinstance(result, dict) and not result.get("error")][:3]
print("\n=== additional validated examples ===")
for example in example_people:
    example_rec = all_recs[example]
    print(f"\n{example}: {example_rec['total_grams']}g total")
    for line in example_rec.get("recommendations", []):
        print(f"  {line['item']}: {line['count']} {line['unit']} ({line['grams']}g)")


## Step 5 - Recommendation vs. past consumption

Compare the actual cached LLM serving suggestion with this student's average
amount taken across the historical meals.

In [ ]:
from visualize_outputs import _recommendation_history
from granica.config import STUDENT_HISTORY_JSON

history_db = load_json(STUDENT_HISTORY_JSON)
example_recs = {name: all_recs[name] for name in example_people}
charts = _recommendation_history(example_recs, history_db)
print("historical meals:", len(history.get("consumption", [])))
for path in charts:
    display(Image.open(path))

# Also show the generated box+mask composite for every held-out test image.
from visualize_outputs import _grounding_images
from granica.grounding import load_groundings
from granica.segmentation import load_segmentation
all_overlays = _grounding_images(load_groundings(), load_segmentation())
test_split_name = "dinner-2026-09-25-test"
for path in all_overlays:
    if Path(path).parent.parent.name == test_split_name:
        display(Image.open(path))


## Summary

The steps above show train-only face comparisons, all held-out test plates,
grounding boxes and labeled SAM overlays, weight predictions, several validated
recommendations, and history-alignment charts. Recommendations are closed-set:
every item is on tonight's menu and every unit has a fixed gram value.
